# 10 — Évaluation du retrieval et de la génération

Regroupe l'audit du golden set, Recall/Precision/Hit Rate/MRR, latences et revue humaine des réponses. Les métriques séparent désormais l'URL/page attendue (diagnostic de retrieval) de l'étayage factuel par une autre citation officielle, et distinguent route exacte de route `all` effectivement couvrante.

> Vérifiez toujours les drapeaux `RUN_*` avant l'exécution. Pour cette reconstruction propre, les traitements locaux nécessaires des étapes 02 à 07 sont déjà activés ; les collectes réseau et évaluations GPU facultatives restent désactivées.


In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.yaml').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Impossible de trouver config.yaml et src/.')

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Projet :', PROJECT_ROOT)
print('Python :', sys.executable)
print('Données actives :', PROJECT_ROOT / 'data')


In [ ]:
import re
import pandas as pd
from config import load_config
from evaluation import load_golden_set, validate_golden_set

config = load_config(PROJECT_ROOT / 'config.yaml')
examples = load_golden_set(config.evaluation.expanded_golden_set_full_path)
chunks_path = config.project.data_path / 'chunks' / 'chunks.jsonl'
corpus_urls = []
for line in chunks_path.read_text(encoding='utf-8').splitlines():
    if line.strip():
        corpus_urls.append(__import__('json').loads(line).get('url', ''))
report = validate_golden_set(examples, corpus_urls=corpus_urls)
display(pd.Series(report).to_frame('valeur'))
display(pd.DataFrame([example.__dict__ for example in examples]).groupby(['split', 'category']).size().rename('questions').to_frame())


In [ ]:
import json

local_records = []
for path in sorted(config.local_documents.raw_full_path.glob('local_pdf_*.json')):
    local_records.append(json.loads(path.read_text(encoding='utf-8')))
economic_chunks_path = config.local_documents.chunks_full_path
economic_urls = []
if economic_chunks_path.exists():
    for line in economic_chunks_path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            economic_urls.append(json.loads(line).get('url', ''))
indexed_urls = set(economic_urls)
missing_local_urls = [record['url'] for record in local_records if record['url'] not in indexed_urls]
display(pd.Series({
    'pdf_locaux_importés': len(local_records),
    'pdf_locaux_représentés_dans_les_chunks': len(local_records) - len(missing_local_urls),
    'pdf_locaux_absents_des_chunks': len(missing_local_urls),
}).to_frame('valeur'))
assert not missing_local_urls, missing_local_urls[:10]


In [ ]:
from source_router import route_question

routing_checks = {
    "Quel est le PIB de la Cote d'Ivoire selon l'ANSTAT ?": 'economic',
    "Quels sont les resultats du RGPH ?": 'economic',
    "Quelles sont les conditions d'admission a l'ENSEA ?": 'institutional',
    "Quels sont les finalistes du concours du CDR cette annee ?": 'facebook',
}
for question, expected_route in routing_checks.items():
    decision = route_question(question)
    print(question, '->', decision.route, '|', decision.reason)
    assert decision.route == expected_route, (question, decision)


## Métriques

- **Recall@k** : part des sources pertinentes retrouvées.
- **Precision@k** : part de résultats pertinents dans le top k.
- **Hit Rate@k** : présence d'au moins une source pertinente.
- **MRR** : récompense une première bonne source placée très haut.
- **P50/P95** : latence typique et latence haute.
- **Fidélité, correction, complétude** : revue humaine de la génération.


In [ ]:
RUN_RETRIEVAL_EVALUATION = False
if RUN_RETRIEVAL_EVALUATION:
    from rag_pipeline import RAGPipeline
    from evaluation import RetrievalEvaluator
    pipeline = RAGPipeline(config)
    development = [example for example in examples if example.split == 'development']
    evaluator = RetrievalEvaluator(config.evaluation)
    metrics = evaluator.evaluate(pipeline, development, run_id='workflow-development')
    display(pd.Series(metrics).to_frame('valeur'))
    details = pd.DataFrame(evaluator.last_details)
    display(details[details['first_relevant_rank'].isna()][['question', 'top_title', 'top_url', 'relevant_urls']])
else:
    print('Benchmark GPU désactivé. Le notebook benchmark_retrieval conserve les runs historiques.')


In [ ]:
RUN_GENERATION_REVIEW = False
if RUN_GENERATION_REVIEW:
    if 'pipeline' not in globals():
        from rag_pipeline import RAGPipeline
        pipeline = RAGPipeline(config)
    rows = []
    for example in [item for item in examples if item.should_answer][:8]:
        response = pipeline.ask(example.question)
        rows.append({
            'question': example.question,
            'answer': response.answer,
            'answer_words': len(response.answer.split()),
            'answer_mode': pipeline.generator.last_prompt_stats.get('answer_mode'),
            'citations': len(response.citations),
            'mechanical_style': bool(re.search(r'(?im)^\s*(synthèse|réponse|citation)\s*:', response.answer)),
            'human_correct': None,
            'human_faithful': None,
            'human_complete': None,
        })
    display(pd.DataFrame(rows))


## Mesures complémentaires pour les tableaux

Un simple hit sur le bon PDF ne suffit pas. Les évaluations tabulaires suivent désormais :

- le hit sur la page attendue ;
- le hit d'un chunk `table_fact` ;
- la présence de la valeur de référence ;
- la relation correcte entre valeur et année, afin qu'une bonne valeur attribuée à la mauvaise
  période soit comptée comme une erreur.

In [ ]:
from evaluation import is_documentary_abstention, reference_relation_score

metric_examples = pd.DataFrame([
    {
        'question': 'Quel était le rapport de masculinité en 2025 ?',
        'référence': '109,3 hommes pour 100 femmes.',
        'réponse': 'En 2025, le rapport était de 109,3 hommes pour 100 femmes.',
    },
    {
        'question': 'Quel était le rapport de masculinité en 2025 ?',
        'référence': '109,3 hommes pour 100 femmes.',
        'réponse': 'En 2024, le rapport était de 109,3 hommes pour 100 femmes.',
    },
])
metric_examples['score_relation'] = metric_examples.apply(
    lambda row: reference_relation_score(row['question'], row['référence'], row['réponse']),
    axis=1,
)
display(metric_examples)
assert metric_examples.loc[0, 'score_relation'] >= 0.80
assert metric_examples.loc[1, 'score_relation'] < 0.50
print('La métrique distingue désormais une valeur correcte reliée à la bonne année.')